# Test the saved project LoRA adapter on the original APTOS split

This is the standalone version of the test that produced **81.47% accuracy, QWK 0.9286,
and macro ROC-AUC 0.9488** using the project's existing adapter. Those are previous observations,
not precomputed outputs in this notebook. The paper reports 81.74%, 0.9292 and 0.9488.

**No training happens here.** This notebook attaches
`Qwen3/lr0.0005_epoch4_batchsize8/best_lora` to the cached Qwen3-VL base and evaluates exactly
**367 APTOS test images**, using the recovered original split and the original five-answer
log-likelihood scoring method. It does not generate a free-form answer and parse it.

The newer combined split has different APTOS test membership; using that split with the old adapter
would include images marked as training in the original split. This notebook reads the original
split assignments from `Downloads/aptos_dataset_80_10_10.csv` and uses the combined CSV only to locate images.

## Environment and how to run

Use **Python (afterimage_env)** in VS Code/Jupyter. From Anaconda Prompt at the repository root:

```bat
conda activate afterimage_env
python -m pip install notebook ipykernel
python -m ipykernel install --user --name afterimage_env --display-name "Python (afterimage_env)"
```

The project model dependencies must already be installed. Optional plots use `matplotlib`
(`python -m pip install matplotlib` if needed); all predictions and numeric reports work without it.
This notebook never installs or upgrades model packages. Stop/unload other GPU model servers before running.

Run the cells from top to bottom. The kernel only prepares data and launches subprocesses; each GPU process
exits when finished. Do not run both notebooks simultaneously on the same GPU. For unattended runs, keep
the computer powered, disable sleep for that session, and keep Jupyter/the notebook kernel running.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid
import pandas as pd
from IPython.display import display

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "finetuning/Qwen3/paper_reproduction.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter inside the Afterimage checkout.")
QWEN_DIR = REPO_ROOT / "finetuning/Qwen3"
if str(QWEN_DIR) not in sys.path:
    sys.path.insert(0, str(QWEN_DIR))
import paper_reproduction as reproduction

if Path(sys.prefix).name.casefold() != "afterimage_env" or sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"Select the Python 3.11 afterimage_env kernel. Current Python: {sys.executable}")

ORIGINAL_APTOS_SPLIT = Path(r"C:\Users\moht2\Downloads\aptos_dataset_80_10_10.csv")
DATA_ROOT = Path(r"C:\Users\moht2\Downloads\DR_Aptos_Messidor_Only_Full_Dataset\Res_512")
IMAGE_MAPPING_CSV = DATA_ROOT / "Split_80_10_10/dataset_80_10_10.csv"
IMAGE_DIR = DATA_ROOT / "Aptos_messidor_external_CLAHE"
WORK_ROOT = QWEN_DIR / "artifacts/paper_reproduction"
VERIFY_IMAGES = True
print("Python:", sys.executable)
print("Original split:", ORIGINAL_APTOS_SPLIT)
print("Generated outputs:", WORK_ROOT)


SAVED_ADAPTER = REPO_ROOT / "Qwen3/lr0.0005_epoch4_batchsize8/best_lora"
INCLUDE_EXTERNAL_MESSIDOR = False  # Optional separate 1,057-image report; APTOS always stays 367.
EVAL_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
EVAL_DIR = WORK_ROOT / "saved_adapter_tests" / EVAL_ID
print("Saved adapter:", SAVED_ADAPTER)
print("This evaluation:", EVAL_DIR)

Python: c:\Users\moht2\.conda\envs\afterimage_env\python.exe
Original split: C:\Users\moht2\Downloads\aptos_dataset_80_10_10.csv
Generated outputs: c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction
Saved adapter: c:\Users\moht2\Documents\Afterimage\Qwen3\lr0.0005_epoch4_batchsize8\best_lora
This evaluation: c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction\saved_adapter_tests\20261104T203559Z_47819c


## Recover and verify the original split

The source CSV hash and exact per-class counts are checked before exporting train/validation/test CSVs.
No split is regenerated. Every mapped image must exist; by default every PNG is checked for integrity
and 512-by-512 dimensions. The label-bearing `vlm_prompt` captions are excluded from exported inputs.

In [2]:
DATASET_MANIFEST = reproduction.prepare_data(
    ORIGINAL_APTOS_SPLIT, IMAGE_MAPPING_CSV, IMAGE_DIR, WORK_ROOT, verify_images=VERIFY_IMAGES,
)
manifest = json.loads(DATASET_MANIFEST.read_text(encoding="utf-8"))
counts = pd.DataFrame([
    {"partition": split, "source": "APTOS", "images": manifest["counts"][split]}
    for split in ("train", "val", "test")
] + [{"partition": "external test", "source": "Messidor", "images": manifest["counts"]["messidor_external"]}])
display(counts)
display(pd.DataFrame(reproduction.APTOS_COUNTS, index=pd.Index(range(5), name="grade")).T)
print("Original APTOS test: 367 images. Messidor is a separate external cohort of 1,057 images.")
print("Original CSVs and image files have not been modified.")

Verified 1000/4719 image files
Verified 2000/4719 image files
Verified 3000/4719 image files
Verified 4000/4719 image files
Verified 4719/4719 image files


,partition,source,images
0,train,APTOS,2929
1,val,APTOS,366
2,test,APTOS,367
3,external test,Messidor,1057


grade,0,1,2,3,4
train,1444,296,799,154,236
val,180,37,100,20,29
test,181,37,100,19,30


Original APTOS test: 367 images. Messidor is a separate external cohort of 1,057 images.
Original CSVs and image files have not been modified.


## Check the saved adapter and cached base model

In [3]:
# Pin the exact local base snapshot used in the successful saved-adapter check.
# local_files_only=True prevents an unattended download or silent model revision change.
BASE_SNAPSHOT = reproduction.resolve_base_snapshot()
print("Cached base snapshot:", BASE_SNAPSHOT)
display(pd.Series(reproduction.package_versions(), name="installed version"))


adapter_dir, adapter_weights = reproduction.validate_adapter(SAVED_ADAPTER)
print("Adapter weights:", adapter_weights)
print("Adapter SHA-256:", reproduction.sha256_file(adapter_weights))

c:\Users\moht2\.conda\envs\afterimage_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cached base snapshot: C:\Users\moht2\.cache\huggingface\hub\models--unsloth--Qwen3-VL-8B-Instruct-unsloth-bnb-4bit\snapshots\b5b904c3fcdc7541adf2a2bb219b0ed95288c794


torch           2.14.1+cu130
torchvision     0.29.1+cu130
unsloth           2026.10.10
unsloth_zoo       2026.10.10
transformers          5.17.0
trl                   1.15.0
peft                  0.21.2
datasets               4.8.5
numpy                  2.4.6
pandas                 3.0.6
bitsandbytes          0.50.2
Name: installed version, dtype: str

Adapter weights: C:\Users\moht2\Documents\Afterimage\Qwen3\lr0.0005_epoch4_batchsize8\best_lora\adapter_model.safetensors
Adapter SHA-256: ec31a623353ea3f2209ca39404ced725d4dd389a7af9d32e5238a039d378d183


## Run the test

This launches a fresh process using this kernel's Python. The FP16 inference setup, text-before-image
prompt, five candidate answers, prompt masking and total answer likelihood follow the original notebook.
Scores `p0` through `p4` are normalized among the five candidate strings; they are not calibrated clinical probabilities.

The full 367-image check previously took about six minutes on this machine. A separately enabled
Messidor test adds time. Predictions are flushed after each image; rerunning this execution cell with
the same `EVAL_DIR` resumes a valid partial file, or keeps completed results after identity checks.

In [4]:
EVAL_COMMAND = [
    sys.executable, "-u", QWEN_DIR / "paper_reproduction.py", "evaluate",
    "--adapter", adapter_dir, "--base-snapshot", BASE_SNAPSHOT,
    "--test-csv", manifest["csv_paths"]["test"], "--output-dir", EVAL_DIR,
]
if INCLUDE_EXTERNAL_MESSIDOR:
    EVAL_COMMAND.extend(["--external-csv", manifest["csv_paths"]["messidor_external"]])
reproduction.run_live(EVAL_COMMAND, EVAL_DIR / "evaluation.log")

c:\Users\moht2\.conda\envs\afterimage_env\python.exe -u c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\paper_reproduction.py evaluate --adapter C:\Users\moht2\Documents\Afterimage\Qwen3\lr0.0005_epoch4_batchsize8\best_lora --base-snapshot C:\Users\moht2\.cache\huggingface\hub\models--unsloth--Qwen3-VL-8B-Instruct-unsloth-bnb-4bit\snapshots\b5b904c3fcdc7541adf2a2bb219b0ed95288c794 --test-csv C:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction\data\4fda605471d5430c3733\test.csv --output-dir c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction\saved_adapter_tests\20261104T203559Z_47819c
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1104 23:36:11.862000 42708 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error i

## Inspect metrics, per-class results and plots

In [5]:
def show_evaluation(report_dir, name="aptos_test"):
    report_dir = Path(report_dir)
    metrics = json.loads((report_dir / f"{name}.summary.json").read_text(encoding="utf-8"))
    columns = ["samples", "accuracy", "qwk", "macro_ovr_auc", "macro_f1", "balanced_accuracy"]
    display(pd.DataFrame([{key: metrics[key] for key in columns}], index=[name]))
    display(pd.DataFrame(metrics["per_class"]))
    matrix = pd.DataFrame(metrics["confusion_matrix"], index=pd.Index(range(5), name="true_grade"), columns=range(5))
    display(matrix)
    try:
        import matplotlib.pyplot as plt
        import numpy as np
    except ImportError:
        print("Numeric reports are complete. Install matplotlib only if you want plots.")
        return metrics
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(matrix.to_numpy(), cmap="Blues")
    fig.colorbar(im, ax=ax)
    ax.set(xticks=range(5), yticks=range(5), xlabel="Predicted grade", ylabel="True grade", title=name)
    for i in range(5):
        for j in range(5):
            value = int(matrix.iloc[i, j])
            ax.text(j, i, str(value), ha="center", va="center", color="white" if value > matrix.to_numpy().max()/2 else "black")
    fig.tight_layout()
    fig.savefig(report_dir / f"{name}.confusion_matrix.png", dpi=180)
    plt.show()
    predictions = pd.read_csv(report_dir / f"{name}.predictions.csv")
    fig, ax = plt.subplots(figsize=(6, 5))
    for grade, label in enumerate(reproduction.LABEL_NAMES):
        truth = predictions["y_true"].to_numpy() == grade
        scores = predictions[f"p{grade}"].to_numpy()
        order = np.argsort(-scores, kind="stable")
        y, s = truth[order], scores[order]
        last = np.r_[np.flatnonzero(np.diff(s)), len(s)-1]
        tp = np.cumsum(y)[last]
        fp = 1 + last - tp
        ax.plot(np.r_[0, fp]/(~truth).sum(), np.r_[0, tp]/truth.sum(),
                label=f"{grade}: {label} (AUC={metrics['per_class'][grade]['auc']:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set(xlabel="False positive rate", ylabel="True positive rate", title=f"{name}: one-vs-rest ROC")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(report_dir / f"{name}.roc.png", dpi=180)
    plt.show()
    return metrics

In [6]:
aptos_metrics = show_evaluation(EVAL_DIR, "aptos_test")
assert aptos_metrics["samples"] == 367
comparison = pd.DataFrame([
    {"evaluation": "Paper (reported)", "accuracy": 0.8174, "qwk": 0.9292, "macro_ovr_auc": 0.9488},
    {"evaluation": "Previous saved-adapter check", "accuracy": 0.8147138964577657,
     "qwk": 0.928550335012589, "macro_ovr_auc": 0.9488104097764621},
    {"evaluation": "This notebook execution", **{key: aptos_metrics[key] for key in ("accuracy", "qwk", "macro_ovr_auc")}},
])
display(comparison)
comparison.to_csv(EVAL_DIR / "comparison_to_paper.csv", index=False)
if INCLUDE_EXTERNAL_MESSIDOR:
    external_metrics = show_evaluation(EVAL_DIR, "messidor_external")
    assert external_metrics["samples"] == 1057
print("Saved outputs:", EVAL_DIR)

,samples,accuracy,qwk,macro_ovr_auc,macro_f1,balanced_accuracy
aptos_test,367,0.814714,0.92855,0.94881,0.682262,0.698733


,grade,label,support,precision,recall,f1,auc
0,0,No DR,181,0.988764,0.972376,0.980501,0.997327
1,1,Mild,37,0.675676,0.675676,0.675676,0.953563
2,2,Moderate,100,0.786517,0.700000,0.740741,0.929551
3,3,Severe,19,0.275000,0.578947,0.372881,0.904265
4,4,Proliferative DR,30,0.739130,0.566667,0.641509,0.959347


,0,1,2,3,4
true_grade,,,,,
0,176,4,1,0,0
1,2,25,10,0,0
2,0,8,70,20,2
3,0,0,4,11,4
4,0,0,4,9,17


Numeric reports are complete. Install matplotlib only if you want plots.


,evaluation,accuracy,qwk,macro_ovr_auc
0,Paper (reported),0.817400,0.92920,0.94880
1,Previous saved-adapter check,0.814714,0.92855,0.94881
2,This notebook execution,0.814714,0.92855,0.94881


Saved outputs: c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\paper_reproduction\saved_adapter_tests\20261104T203559Z_47819c


## What is saved

The output folder contains incremental per-image predictions and five candidate scores, a summary JSON,
per-class metrics, confusion-matrix CSV, optional plots, the comparison table, streamed logs and an
evaluation manifest with adapter/data/helper hashes, the exact base snapshot and package versions.
The separate GPU process has exited, releasing its model memory.

The current environment differs from the original Unsloth 2026.2.1 / Torch 2.10.0 /
Transformers 5.3.0.dev0 stack. The recovered split matches the original image IDs and labels;
the historical preprocessed image bytes are unavailable for a byte-for-byte comparison.
These limits explain why this is a controlled reproduction attempt rather than a promise of identical numbers.

References: `Downloads/Qwen3_VL_8B_DR_finetuningAndEvaluation.ipynb` and the repository's original
APTOS orchestrator contain the full-answer scoring function and the reported 0.8174 / 0.9292 / 0.9488 outputs.
`Untitled39.ipynb` consolidates those prediction reports. `Untitled36.ipynb` contains additional
augmentation experiments; those training settings are not used here.